# RSRP около культурных объектов Вены

Исходные данные: `vienna_drive_test.csv`, операторы A, B, C. Основной показатель — RSRP в dBm. Чем значение ближе к нулю, тем сильнее сигнал.

**Запуск в Google Colab:** разместите папку проекта в файловой системе среды рядом с notebook. Если нужно, перейдите в неё командой `%cd /content/GeoProject` (укажите свой путь), затем выберите «Среда выполнения → Выполнить всё». Первая ячейка найдёт уже размещённые файлы проекта. CSV читается из `data/`, `../data/` или из `deliverables/submission.zip`; распаковывать архив вручную не нужно.

Локально откройте notebook из папки репозитория. Сохранённый `output/geocoding.json` позволяет повторить расчёты без API-ключа. Только для новых адресов понадобится `YANDEX_API_KEY` в Secrets Colab или локальном `.env`.

Ячейки выполняйте по порядку. Если не хватает данных или геокодирование требует проверки, появится сообщение с дальнейшим действием. Зависимые этапы и экспорт пропускаются, пока проблема не устранена. После исправления повторите «Выполнить всё».

In [ ]:
from pathlib import Path
import sys


def find_project_root():
    here = Path.cwd()
    candidates = [here, here / "project", here / "GeoProject", here.parent / "GeoProject"]
    for candidate in candidates:
        if (candidate / "requirements.txt").is_file() and (
            (candidate / "output").is_dir() or (candidate / "data").is_dir()
            or (candidate / "deliverables" / "submission.zip").is_file()
        ):
            return candidate
    return None


ROOT = find_project_root()
if ROOT is None:
    ROOT = Path.cwd()
    print("Папка проекта не найдена. Разместите рядом requirements.txt и папки данных из репозитория.")
else:
    print("Папка проекта:", ROOT)

# Проверяем совместимость версий при каждом запуске; подходящие установленные версии сохраняются.
%pip install -q 'numpy>=2.1,<3' 'pandas>=2.2.2,<3' 'geopandas>=1,<1.2' 'h3>=4,<5' scipy rasterio requests python-dotenv

## Настройки

Расстояния считаются в UTM 33N. Радиус 250 м задаёт окрестность здания; H3 resolution 9 даёт локальную агрегацию без привязки к районам. Параметры общие для карты и таблиц.

In [ ]:
from pathlib import Path
import json
import os
from zipfile import BadZipFile, ZipFile

from IPython.display import display

import geopandas as gpd
import h3
import numpy as np
import pandas as pd
import rasterio
import requests
from dotenv import load_dotenv
from rasterio.transform import from_origin
from scipy.spatial import cKDTree
from shapely.geometry import Polygon

DATA_CANDIDATES = [ROOT / "data" / "vienna_drive_test.csv", ROOT.parent / "data" / "vienna_drive_test.csv"]
DATA_FILE = next((path for path in DATA_CANDIDATES if path.is_file()), None)
SOURCE_ARCHIVE = ROOT / "deliverables" / "submission.zip"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)
load_dotenv(ROOT.parent / ".env")
load_dotenv(ROOT / ".env")
RADIUS_M = 250
H3_RESOLUTION = 9
PIXEL_M = 100
MIN_COUNT = 3
CRS_METRIC = "EPSG:32633"
BREAKS = [-110, -100, -90, -80]
SIGNAL_LABELS = ["< −110", "−110 … < −100", "−100 … < −90", "−90 … < −80", "≥ −80"]
# Cividis: слабому сигналу соответствует тёмный цвет, сильному — светлый.
SIGNAL_COLORS = ["#00224e", "#434e6c", "#7d7c78", "#bcae6c", "#fee838"]
OPERATOR_COLORS = {"A": "#0072B2", "B": "#E69F00", "C": "#CC79A7"}
OBJECTS = [
    (1, "Technisches Museum", "Mariahilfer Straße 212, 1140 Wien, Austria", "https://www.technischesmuseum.at/besuchen/barrierefreier_besuch"),
    (2, "Raimund Theater", "Wallgasse 18-20, 1060 Wien, Austria", "https://www.musicalvienna.at/de-de/ueber-uns/raimund-theater"),
    (3, "Secession", "Friedrichstraße 12, 1010 Wien, Austria", "https://secession.at/kontakt"),
    (4, "Theater an der Wien", "Linke Wienzeile 6, 1060 Wien, Austria", "https://www.theater-wien.at/de/service/ihr-besuch/1"),
    (5, "Upper Belvedere", "Prinz-Eugen-Straße 27, 1030 Wien, Austria", "https://www.belvedere.at/en/visit/upper-belvedere"),
]


def signal_class(values):
    return np.searchsorted(BREAKS, np.asarray(values), side="right").astype(int)

# Сбрасываем результаты при повторном запуске, чтобы не использовать данные прошлой попытки.
df = objects = points = metric = grid = objects_map = buffers = object_signal = None
audit = {}
raster_audit = None
analysis_checks = {}
analysis_ok = False
step_status = {name: "не выполнен" for name in ["Очистка", "Геокодирование", "Расчёты", "Растр", "Экспорт"]}

## Очистка

Исправляем только однозначную перестановку широты и долготы. Пропуски в дополнительных радиопоказателях не исключают запись. Если один ID содержит разные координаты или RSRP, исключаем все его версии: выбрать правильную по имеющимся данным нельзя.

In [ ]:
def read_source():
    try:
        if DATA_FILE is not None:
            print("Исходные данные:", DATA_FILE)
            return pd.read_csv(DATA_FILE)
        if SOURCE_ARCHIVE.is_file():
            with ZipFile(SOURCE_ARCHIVE) as archive:
                member = next((name for name in ["project/data/vienna_drive_test.csv", "data/vienna_drive_test.csv"]
                               if name in archive.namelist()), None)
                if member is not None:
                    print("Исходные данные:", str(SOURCE_ARCHIVE) + " → " + member)
                    with archive.open(member) as source:
                        return pd.read_csv(source)
    except (OSError, BadZipFile, UnicodeError, pd.errors.ParserError, pd.errors.EmptyDataError):
        print("Не удалось прочитать CSV или архив. Загрузите целый файл заново и повторите запуск.")
        return None
    print("Исходный CSV не найден. Поместите vienna_drive_test.csv в папку data/ или восстановите deliverables/submission.zip.")
    return None


def clean_data(raw):
    df = raw.copy()
    audit = {"input_rows": len(df)}
    required = ["measurement_id", "time", "latitude", "longitude", "rsrp_dbm", "operator"]
    missing = set(required) - set(df.columns)
    if missing:
        print(f"В CSV нет обязательных полей: {', '.join(sorted(missing))}. Выберите исходный vienna_drive_test.csv.")
        return None, audit
    for col in ["measurement_id", "latitude", "longitude", "rsrp_dbm"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    swapped = df.latitude.between(16.1, 16.7) & df.longitude.between(48.0, 48.4)
    audit["swapped_coordinates_fixed"] = int(swapped.sum())
    df.loc[swapped, ["latitude", "longitude"]] = df.loc[swapped, ["longitude", "latitude"]].to_numpy()
    # Прямоугольник охватывает Вену; это проверка координат, не административная граница.
    checks = [
        ("invalid_location_removed", df.latitude.between(48.0, 48.4) & df.longitude.between(16.1, 16.7)),
        ("invalid_rsrp_removed", np.isfinite(df.rsrp_dbm) & df.rsrp_dbm.lt(0)),
        ("missing_required_removed", df[required].notna().all(axis=1) & df.operator.isin(["A", "B", "C"])),
    ]
    for label, valid in checks:
        audit[label] = int((~valid.reindex(df.index)).sum())
        df = df.loc[valid.reindex(df.index)].copy()
    core = ["time", "latitude", "longitude", "rsrp_dbm", "operator"]
    duplicates = df.loc[df.measurement_id.duplicated(False)]
    differences = duplicates.groupby("measurement_id")[core].nunique(dropna=False)
    conflicts = differences.index[differences.gt(1).any(axis=1)]
    audit["conflicting_ids"] = len(conflicts)
    audit["conflicting_id_rows_removed"] = int(df.measurement_id.isin(conflicts).sum())
    # По конфликтному ID нельзя определить, какая версия верна: исключаем обе.
    df = df.loc[~df.measurement_id.isin(conflicts)].copy()
    audit["repeated_ids_collapsed"] = int(df.measurement_id.duplicated().sum())
    df = df.drop_duplicates("measurement_id").reset_index(drop=True)
    df["measurement_id"] = df.measurement_id.astype("int64")
    df["survey_date"] = df.time.dt.strftime("%Y-%m-%d")
    df["signal_class"] = signal_class(df.rsrp_dbm)
    audit["output_rows"] = len(df)
    if df.empty:
        print("После очистки не осталось измерений. Проверьте координаты, время, RSRP и обозначения операторов A/B/C.")
        return None, audit
    print("Строк после очистки:", len(df))
    return df, audit

df = None
audit = {}
raw = read_source()
if raw is not None:
    df, audit = clean_data(raw)
    del raw
step_status["Очистка"] = "готово" if df is not None else "нужен корректный исходный CSV"
if audit:
    display(pd.Series(audit, name="Число записей").to_frame())

## Геокодирование Яндексом

Используем официальные адреса пяти культурных объектов. Они выбраны после предварительной проверки наличия измерений рядом с ними. Сначала проверяем сохранённый кэш: адреса, точность и координаты должны соответствовать текущему списку объектов. При корректном кэше сеть и API-ключ не нужны.

Если кэш отсутствует или не подходит, пробуем получить новые ответы Яндекса. Без ключа, при сетевой ошибке или сомнительном ответе выводим пояснение и пропускаем зависимые этапы. Неполный результат не заменяет кэш. Сохраняем адрес запроса и ответ, но не ключ. Точка адреса представляет окрестность объекта, а не измерение внутри здания.

In [ ]:
def valid_geocoding(saved):
    if not isinstance(saved, list) or any(not isinstance(row, dict) for row in saved):
        return False
    expected = [(item[0], item[1], item[2]) for item in OBJECTS]
    if [(row.get("object_id"), row.get("name"), row.get("address")) for row in saved] != expected:
        return False
    for row in saved:
        try:
            lon, lat = float(row["longitude"]), float(row["latitude"])
        except (KeyError, TypeError, ValueError):
            return False
        if (row.get("precision") not in ["exact", "number"]
                or not (16.1 <= lon <= 16.7 and 48.0 <= lat <= 48.4)
                or not all(row.get(field) for field in ["matched_address", "source_url", "geocoded_at"])):
            return False
    return True


def yandex_key():
    key = os.environ.get("YANDEX_API_KEY", "").strip()
    if not key and "google.colab" in sys.modules:
        try:
            from google.colab import userdata
            key = userdata.get("YANDEX_API_KEY")
        except Exception:
            # У Colab разные исключения для отсутствующего секрета и запрета доступа.
            print("Секрет YANDEX_API_KEY недоступен. Для нового геокодирования добавьте его в Secrets и разрешите доступ.")
    return key


def request_geocoding(key):
    saved = []
    for object_id, name, address, source_url in OBJECTS:
        try:
            response = requests.get("https://geocode-maps.yandex.ru/v1/", params={
                "apikey": key, "geocode": address, "format": "json", "lang": "en_US", "results": 1,
            }, timeout=45)
        except requests.RequestException:
            print(f"Нет связи с Яндекс Геокодером ({name}). Проверьте интернет и повторите запуск.")
            return None
        if response.status_code != 200:
            print(f"Яндекс Геокодер: HTTP {response.status_code}, объект {name}. Проверьте API-ключ, квоту и доступ к сервису.")
            return None
        try:
            payload = response.json()
            matches = payload["response"]["GeoObjectCollection"]["featureMember"]
            if not matches:
                print(f"Адрес не найден: {name}. Уточните его в списке OBJECTS.")
                return None
            obj = matches[0]["GeoObject"]
            meta = obj["metaDataProperty"]["GeocoderMetaData"]
            lon, lat = map(float, obj["Point"]["pos"].split())
            saved.append({"object_id": object_id, "name": name, "address": address,
                          "source_url": source_url, "longitude": lon, "latitude": lat,
                          "matched_address": meta["text"], "precision": meta.get("precision", "unknown"),
                          "geocoded_at": pd.Timestamp.now(tz="UTC").isoformat(), "response": payload})
        except (KeyError, IndexError, TypeError, ValueError):
            print(f"Ответ геокодера не удалось прочитать: {name}. Повторите запрос позднее.")
            return None
    if not valid_geocoding(saved):
        print("Проверьте ответы геокодера: нужны адресная точность exact/number и координаты в пределах Вены.")
        return None
    return saved


def geocode_objects():
    cache = OUT / "geocoding.json"
    saved = None
    if cache.is_file():
        try:
            candidate = json.loads(cache.read_text(encoding="utf-8"))
        except (OSError, UnicodeError, json.JSONDecodeError):
            candidate = None
        if valid_geocoding(candidate):
            saved = candidate
            print("Используем проверенный кэш геокодирования; API-ключ не нужен.")
        else:
            print("Кэш геокодирования повреждён или не соответствует адресам. Нужен исходный кэш либо новый запрос.")
    if saved is None:
        key = yandex_key()
        if not key:
            print("Геокодирование пропущено. Восстановите output/geocoding.json из репозитория или добавьте YANDEX_API_KEY в .env / Secrets Colab.")
            return None
        saved = request_geocoding(key)
        if saved is None:
            return None
        try:
            cache.write_text(json.dumps(saved, ensure_ascii=False, indent=2), encoding="utf-8")
        except OSError:
            print("Не удалось сохранить кэш. В этом запуске используем проверенный ответ в памяти.")
    frame = pd.DataFrame([{k: v for k, v in row.items() if k != "response"} for row in saved])
    frame[["longitude", "latitude"]] = frame[["longitude", "latitude"]].astype(float)
    display(frame[["name", "matched_address", "precision"]])
    return gpd.GeoDataFrame(frame, geometry=gpd.points_from_xy(frame.longitude, frame.latitude), crs=4326)


objects = geocode_objects()
step_status["Геокодирование"] = "готово" if objects is not None else "нужен проверенный кэш или ответ Яндекса"

## Пространственные расчёты

Медиана RSRP в 250 м считается отдельно для каждого оператора, минимум по трём записям. Медиана уменьшает влияние отдельных экстремальных значений. H3-медиана и число измерений показывают пространственное распределение сигнала и обеспеченность данными. Значения в dBm используются как описательная статистика; это не усреднение мощности в линейных единицах.

In [ ]:
def spatial_analysis(df, objects):
    points = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs=4326)
    metric = points.to_crs(CRS_METRIC)
    object_metric = objects.to_crs(CRS_METRIC)
    df["h3_id"] = [h3.latlng_to_cell(lat, lon, H3_RESOLUTION) for lat, lon in zip(df.latitude, df.longitude)]
    points["h3_id"] = df.h3_id
    metric["h3_id"] = df.h3_id
    summary = df.groupby(["h3_id", "operator"]).agg(
        median_rsrp_dbm=("rsrp_dbm", "median"), measurement_count=("measurement_id", "size")
    ).reset_index()
    summary["signal_class"] = signal_class(summary.median_rsrp_dbm)
    polygons = {cell: Polygon([(lon, lat) for lat, lon in h3.cell_to_boundary(cell)]) for cell in summary.h3_id.unique()}
    grid = gpd.GeoDataFrame(summary, geometry=summary.h3_id.map(polygons), crs=4326)
    records = []
    for operator in ["A", "B", "C"]:
        subset = metric.loc[metric.operator.eq(operator)]
        tree = cKDTree(np.column_stack((subset.geometry.x, subset.geometry.y)))
        for obj in object_metric.itertuples():
            centre = [obj.geometry.x, obj.geometry.y]
            idx = tree.query_ball_point(centre, RADIUS_M)
            nearby = subset.iloc[idx]
            median = float(nearby.rsrp_dbm.median()) if len(nearby) >= MIN_COUNT else None
            records.append({"object_id": obj.object_id, "name": obj.name, "address": obj.address,
                            "operator": operator, "median_rsrp_dbm": median,
                            "measurement_count": len(nearby), "nearest_distance_m": float(tree.query(centre)[0]),
                            "status": "достаточно данных" if median is not None else "недостаточно данных",
                            "radius_m": RADIUS_M})
    object_signal = pd.DataFrame(records)
    buffers = object_metric.copy()
    buffers.geometry = object_metric.geometry.buffer(RADIUS_M)
    # Для карты A сохраняем статистику в атрибутах самих объектов и буферов.
    main = object_signal.loc[object_signal.operator.eq("A"), ["object_id", "operator", "median_rsrp_dbm", "measurement_count", "nearest_distance_m", "status"]]
    objects_map = objects.merge(main, on="object_id")
    buffers = buffers.merge(main, on="object_id")
    print(object_signal[["name", "operator", "median_rsrp_dbm", "measurement_count"]].to_string(index=False))
    return points, metric, grid, objects_map, buffers, object_signal

points = metric = grid = objects_map = buffers = object_signal = None
analysis_checks = {}
analysis_ok = False
if df is not None and objects is not None:
    points, metric, grid, objects_map, buffers, object_signal = spatial_analysis(df, objects)
    analysis_checks = {
        "Уникальные ID измерений": bool(df.measurement_id.is_unique),
        "RSRP конечный и отрицательный": bool((np.isfinite(df.rsrp_dbm) & df.rsrp_dbm.lt(0)).all()),
        "Число измерений сохранено при агрегации": bool(grid.measurement_count.sum() == len(df)),
        "Пять объектов с уникальными ID": len(objects) == 5 and bool(objects.object_id.is_unique),
        "15 строк: пять объектов × три оператора": len(object_signal) == 15,
        "Для каждого объекта представлены A, B, C": bool(object_signal.groupby("object_id").operator.nunique().eq(3).all()),
        "При достаточном числе записей рассчитана медиана": bool(object_signal.loc[
            object_signal.measurement_count >= MIN_COUNT, "median_rsrp_dbm"].notna().all()),
    }
    analysis_ok = all(analysis_checks.values())
    step_status["Расчёты"] = "готово" if analysis_ok else "проверки не пройдены; экспорт отключён"
else:
    step_status["Расчёты"] = "пропущен: нужны измерения и координаты объектов"
    print(step_status["Расчёты"])

## Ограниченная оценка между маршрутами

Для карты A в центре каждого пикселя 100 м находим ближайшее измерение из H3-ячейки с минимум тремя записями. Присваиваем медиану его ячейки. Если расстояние больше 250 м, оставляем NoData. Это оценка ближайшим соседом, без физической модели распространения сигнала. Размер пикселя не означает точность измерения 100 м.

In [ ]:
def estimate_raster(metric, grid):
    operator = "A"
    eligible = grid.loc[grid.operator.eq(operator) & grid.measurement_count.ge(MIN_COUNT)]
    supports = metric.loc[metric.operator.eq(operator) & metric.h3_id.isin(eligible.h3_id)]
    if supports.empty:
        print("Растр пропущен: для оператора A нет H3-ячеек с достаточным числом измерений.")
        return None
    coordinates = np.column_stack((supports.geometry.x, supports.geometry.y))
    tree = cKDTree(coordinates)
    xmin, ymin, xmax, ymax = metric.total_bounds
    xmin = np.floor((xmin - RADIUS_M) / PIXEL_M) * PIXEL_M
    ymax = np.ceil((ymax + RADIUS_M) / PIXEL_M) * PIXEL_M
    width = int(np.ceil((xmax + RADIUS_M - xmin) / PIXEL_M))
    height = int(np.ceil((ymax - ymin + RADIUS_M) / PIXEL_M))
    xs = xmin + (np.arange(width) + 0.5) * PIXEL_M
    ys = ymax - (np.arange(height) + 0.5) * PIXEL_M
    xx, yy = np.meshgrid(xs, ys)
    distance, nearest = tree.query(np.column_stack((xx.ravel(), yy.ravel())))
    cell_medians = eligible.set_index("h3_id").median_rsrp_dbm
    supported_values = supports.h3_id.map(cell_medians).to_numpy()
    values = supported_values[nearest].astype("float32")
    valid = distance <= RADIUS_M
    values[~valid] = -9999
    if not valid.any():
        print("Растр пропущен: ни один пиксель не находится в пределах заданного радиуса от измерений.")
        return None
    if not np.isfinite(values[valid]).all():
        print("Растр требует проверки: найдены нечисловые значения сигнала. Экспорт отключён.")
        return None
    path = OUT / "signal_A.tif"
    with rasterio.open(path, "w", driver="GTiff", height=height, width=width, count=1,
                       dtype="float32", crs=CRS_METRIC, transform=from_origin(xmin, ymax, PIXEL_M, PIXEL_M),
                       nodata=-9999, compress="deflate") as dst:
        dst.write(values.reshape(height, width), 1)
        dst.update_tags(operator=operator, method="nearest supporting measurement -> H3 median",
                        max_distance_m=str(RADIUS_M), h3_resolution=str(H3_RESOLUTION), min_count=str(MIN_COUNT))
    return {"raster_valid_pixels": int(valid.sum()), "raster_nodata_pixels": int((~valid).sum()),
            "raster_max_support_distance_m": float(distance[valid].max()), "raster_pixel_m": PIXEL_M}

raster_audit = None
if analysis_ok:
    raster_audit = estimate_raster(metric, grid)
    if raster_audit is not None:
        audit.update(raster_audit)
        print("Растр готов. Пикселей с оценкой:", raster_audit["raster_valid_pixels"])
step_status["Растр"] = "готово" if raster_audit is not None else "пропущен: нет проверенных данных для растра"

## Экспорт

GeoPackage и GeoTIFF используются в QGIS, CSV — в Superset. Карта открывается на операторе A, у которого больше всего измерений. В Superset доступны все три оператора. Периоды и маршруты измерений различаются, поэтому таблицы не являются контролируемым сравнением операторов.

In [ ]:
def export_data(df, points, grid, objects, buffers, object_signal, audit):
    df.to_csv(OUT / "measurements.csv", index=False)
    grid.drop(columns="geometry").to_csv(OUT / "h3_signal.csv", index=False)
    object_signal.to_csv(OUT / "object_signal.csv", index=False)
    objects.drop(columns="geometry").to_csv(OUT / "objects.csv", index=False)
    audit["operator_counts"] = df.operator.value_counts().sort_index().to_dict()
    audit["date_min"] = df.time.min().isoformat()
    audit["date_max"] = df.time.max().isoformat()
    audit["h3_rows"] = len(grid)
    audit["h3_resolution"] = H3_RESOLUTION
    audit["radius_m"] = RADIUS_M
    audit["object_rows"] = len(object_signal)
    gpkg = OUT / "vienna.gpkg"
    if gpkg.exists():
        gpkg.unlink()
    map_fields = ["measurement_id", "time", "survey_date", "operator", "rsrp_dbm", "signal_class", "h3_id", "geometry"]
    points.loc[points.operator.eq("A"), map_fields].to_file(gpkg, layer="measurements", driver="GPKG")
    grid.loc[grid.operator.eq("A")].to_file(gpkg, layer="h3_signal", driver="GPKG")
    objects.to_file(gpkg, layer="cultural_objects", driver="GPKG")
    buffers.to_file(gpkg, layer="matching_buffers", driver="GPKG")
    # Готовые распределения позволяют Superset не сканировать большой CSV при каждом открытии.
    distribution = df.assign(rsrp_bin=(np.floor(df.rsrp_dbm / 5) * 5).astype(int)).groupby(
        ["operator", "rsrp_bin"]).size().rename("measurement_count").reset_index()
    distribution.to_csv(OUT / "rsrp_distribution.csv", index=False)
    daily = df.groupby(["survey_date", "operator"]).size().rename("measurement_count").reset_index()
    daily.to_csv(OUT / "daily_counts.csv", index=False)
    (OUT / "audit.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Результаты:", OUT)

if analysis_ok and raster_audit is not None:
    export_data(df, points, grid, objects_map, buffers, object_signal, audit)
    step_status["Экспорт"] = "готово"
else:
    step_status["Экспорт"] = "пропущен: сначала исправьте входные данные и повторите запуск"
    print("Экспорт пропущен. Существующие CSV и GeoPackage не обновлялись.")

## Проверка и итог запуска

Показываем результаты проверок и состояние каждого этапа. CSV и GeoPackage экспортируются только после успешных проверок расчётов и построения растра. Если этап пропущен, ранее сохранённые файлы не являются результатом текущего запуска.

Частые повторы координат отражают в том числе остановки автомобиля; число записей не равно числу независимых мест измерения.

In [ ]:
if analysis_checks:
    display(pd.DataFrame({
        "Проверка": list(analysis_checks),
        "Результат": ["OK" if passed else "нужна проверка" for passed in analysis_checks.values()],
    }))
else:
    print("Проверки расчётов не выполнялись: сначала подготовьте измерения и координаты объектов.")

display(pd.DataFrame(step_status.items(), columns=["Этап", "Состояние"]))
if step_status["Экспорт"] == "готово":
    print("Расчёты и экспорт завершены. Результаты сохранены в:", OUT)
else:
    print("Запуск неполный. Следуйте сообщениям выше и повторите «Выполнить всё».")